# Étape 5 — Phase 2 : premiers entraînements et test de contrôle

**Objectif :** vérifier, avant de lancer toute la grille d'expériences de RQ2, que **le graphe expert aide vraiment** le knowledge tracing sur Junyi.

On entraîne deux modèles :
- **DKT** : le modèle de référence classique, sans graphe ;
- **GraphKT** : le même principe, mais chaque exercice reçoit aussi l'information de ses prérequis et de ses successeurs dans le graphe.

GraphKT est entraîné avec trois graphes : **expert** (G\*), **aucun** et **aléatoire** (autant de flèches que G\*, placées au hasard).

On répète dans deux régimes :
- **100 %** des élèves d'entraînement ;
- **10 %** des élèves d'entraînement. C'est souvent quand les données sont rares qu'un graphe aide le plus.

**Lecture attendue :** si *expert > aucun* et *expert > aléatoire*, le graphe porte une information utile, et RQ2 a du sens.

**Avant de commencer :** *Exécution → Modifier le type d'exécution → GPU T4*. Durée totale : environ 30 à 60 min.

In [ ]:
# Cellule 1 — Drive, GPU, paramètres
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, accuracy_score

DOSSIER   = '/content/drive/MyDrive/projet_kt'
PREP      = os.path.join(DOSSIER, 'junyi_prepare')
RESULTATS = os.path.join(DOSSIER, 'resultats')
os.makedirs(RESULTATS, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Appareil :', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '(pas de GPU : ce sera lent)')

HP = dict(dim=64, hidden=128, couches_graphe=2, dropout=0.2,
          lr=1e-3, batch=64, epoques_max=20, patience=3)
GRAINE = 42
print(HP)

In [ ]:
# Cellule 2 — Chargement des séquences et du graphe expert
exos = pd.read_csv(os.path.join(PREP, 'exercices.csv'))
N = len(exos)                                   # nombre d'exercices (identifiants 0..N-1)
A_expert = np.load(os.path.join(PREP, 'adjacence_expert.npy')).astype(np.float32)   # A[i, j] = 1 : i prérequis de j
aretes_expert = list(zip(*np.nonzero(A_expert)))

def charger(partie):
    df = pd.read_csv(os.path.join(PREP, f'sequences_{partie}.csv'))
    q = [np.array(s.split(','), dtype=np.int64) for s in df['exercices']]
    r = [np.array(s.split(','), dtype=np.int64) for s in df['reponses']]
    return df['uid'].values, q, r

donnees = {p: charger(p) for p in ['train', 'valid', 'test']}
for p, (u, q, r) in donnees.items():
    print(f'{p:5s} : {len(set(u)):6,} élèves, {len(q):6,} séquences, {sum(map(len, q)):9,} réponses')
print(f'Exercices : {N} | flèches expertes : {len(aretes_expert)}')

In [ ]:
# Cellule 3 — Mise en tenseurs (complétion à droite + masque)
def tenseurs(q_list, r_list):
    L = max(map(len, q_list))
    Q = np.zeros((len(q_list), L), dtype=np.int64)
    R = np.zeros((len(q_list), L), dtype=np.int64)
    M = np.zeros((len(q_list), L), dtype=bool)
    for i, (q, r) in enumerate(zip(q_list, r_list)):
        Q[i, :len(q)], R[i, :len(r)], M[i, :len(q)] = q, r, True
    return torch.from_numpy(Q), torch.from_numpy(R), torch.from_numpy(M)

def sous_ensemble_train(fraction, graine=GRAINE):
    # Fraction des ÉLÈVES d'entraînement (et non des séquences), pour rester cohérent avec le découpage par élève
    u, q, r = donnees['train']
    if fraction >= 1:
        return q, r
    rng = np.random.default_rng(graine)
    gardes = set(rng.choice(np.unique(u), size=max(1, int(fraction * len(np.unique(u)))), replace=False))
    idx = [i for i, x in enumerate(u) if x in gardes]
    return [q[i] for i in idx], [r[i] for i in idx]

T_valid = tenseurs(*donnees['valid'][1:])
T_test  = tenseurs(*donnees['test'][1:])
print('valid', tuple(T_valid[0].shape), '| test', tuple(T_test[0].shape))

### Les deux modèles

- **DKT** (Piech et al., 2015) : un LSTM lit la suite (exercice, réussite) et prédit la réussite du prochain exercice.
- **GraphKT** : avant le LSTM, la représentation de chaque exercice est enrichie par **deux canaux séparés**, l'un venant de ses **prérequis**, l'autre de ses **successeurs**. Les deux canaux étant distincts, le modèle est sensible au **sens** des flèches, ce qui est indispensable pour étudier l'effet des inversions (E3).
  Avec un graphe vide, GraphKT redevient un DKT à représentations d'exercices : c'est la condition « aucun graphe ».

In [ ]:
# Cellule 4 — Modèles
class DKT(nn.Module):
    def __init__(self, n, hp):
        super().__init__()
        self.n = n
        self.emb = nn.Embedding(2 * n, hp['hidden'])
        self.lstm = nn.LSTM(hp['hidden'], hp['hidden'], batch_first=True)
        self.drop = nn.Dropout(hp['dropout'])
        self.out = nn.Linear(hp['hidden'], n)

    def forward(self, q, r):
        h, _ = self.lstm(self.emb(q[:, :-1] + self.n * r[:, :-1]))
        logits = self.out(self.drop(h))                              # (B, L-1, N)
        return logits.gather(2, q[:, 1:].unsqueeze(-1)).squeeze(-1)  # logit du prochain exercice


def normaliser_lignes(A):
    deg = A.sum(1, keepdims=True)
    return np.divide(A, deg, out=np.zeros_like(A), where=deg > 0)


class GraphKT(nn.Module):
    def __init__(self, n, hp, A):
        super().__init__()
        d = hp['dim']
        self.E = nn.Embedding(n, d)
        self.reg_buffers(A)
        self.couches = nn.ModuleList([nn.ModuleDict({'pre': nn.Linear(d, d), 'suc': nn.Linear(d, d)})
                                      for _ in range(hp['couches_graphe'])])
        self.R = nn.Embedding(2, d)
        self.lstm = nn.LSTM(2 * d, hp['hidden'], batch_first=True)
        self.drop = nn.Dropout(hp['dropout'])
        self.pred = nn.Sequential(nn.Linear(hp['hidden'] + d, hp['hidden']), nn.ReLU(), nn.Linear(hp['hidden'], 1))

    def reg_buffers(self, A):
        A = np.asarray(A, dtype=np.float32)
        self.register_buffer('P_pre', torch.from_numpy(normaliser_lignes(A.T.copy())))  # ligne j : moyenne sur les prérequis de j
        self.register_buffer('P_suc', torch.from_numpy(normaliser_lignes(A.copy())))    # ligne i : moyenne sur les successeurs de i

    def representations(self):
        H = self.E.weight
        for c in self.couches:
            H = H + torch.relu(c['pre'](self.P_pre @ H)) * (self.P_pre.sum(1, keepdim=True) > 0) \
                  + torch.relu(c['suc'](self.P_suc @ H)) * (self.P_suc.sum(1, keepdim=True) > 0)
        return H

    def forward(self, q, r):
        H = self.representations()
        x = torch.cat([H[q[:, :-1]], self.R(r[:, :-1])], dim=-1)
        h, _ = self.lstm(x)
        return self.pred(torch.cat([self.drop(h), H[q[:, 1:]]], dim=-1)).squeeze(-1)

print('OK')

### Fonctions qui abîment le graphe (taxonomie E1–E5)

Elles serviront à la grille complète de RQ2. On les définit et on les vérifie dès maintenant. `taux` = proportion du nombre de flèches expertes.

| Code | Erreur | Ce que fait la fonction |
|---|---|---|
| E1 | Fausses flèches aléatoires | ajoute des flèches entre exercices quelconques |
| E2 | Fausses flèches plausibles | ajoute des flèches entre exercices du **même thème** (`topic`), comme le ferait un LLM |
| E3 | Inversions | retourne des flèches A → B en B → A |
| E4 | Suppressions | retire des flèches |
| E5 | Cycles | ajoute C → A quand A mène déjà à C, ce qui crée une boucle |

In [ ]:
# Cellule 5 — Corruption contrôlée du graphe
import networkx as nx

def vers_matrice(aretes, n=N):
    A = np.zeros((n, n), dtype=np.float32)
    for a, b in aretes:
        A[a, b] = 1
    return A

def corrompre(aretes, type_err, taux, graine=GRAINE, topics=exos['topic'].values):
    rng = random.Random(graine)
    E = set(map(tuple, aretes))
    k = int(round(taux * len(E)))
    noeuds = list(range(N))
    if type_err == 'E1':      # ajouts aléatoires
        while k > 0:
            a, b = rng.sample(noeuds, 2)
            if (a, b) not in E and (b, a) not in E:
                E.add((a, b)); k -= 1
    elif type_err == 'E2':    # ajouts plausibles : même thème
        par_topic = {}
        for i, t in enumerate(topics):
            par_topic.setdefault(t, []).append(i)
        groupes = [g for g in par_topic.values() if len(g) >= 2]
        essais = 0
        while k > 0 and essais < 100 * len(E) + 1000:
            essais += 1
            a, b = rng.sample(rng.choice(groupes), 2)
            if (a, b) not in E and (b, a) not in E:
                E.add((a, b)); k -= 1
    elif type_err == 'E3':    # inversions
        for a, b in rng.sample(sorted(E), min(k, len(E))):
            E.discard((a, b)); E.add((b, a))
    elif type_err == 'E4':    # suppressions
        for e in rng.sample(sorted(E), min(k, len(E))):
            E.discard(e)
    elif type_err == 'E5':    # cycles : C -> A avec A ->* C
        G = nx.DiGraph(list(E))
        candidats = sorted(E); rng.shuffle(candidats)
        for a, b in candidats:
            if k == 0:
                break
            desc = list(nx.descendants(G, b)) + [b]
            c = rng.choice(desc)
            if c != a and (c, a) not in E:
                E.add((c, a)); G.add_edge(c, a); k -= 1
    elif type_err == 'aleatoire':   # graphe aléatoire de même taille
        E2 = set()
        while len(E2) < len(aretes):
            a, b = rng.sample(noeuds, 2)
            E2.add((a, b))
        E = E2
    elif type_err == 'aucun':
        E = set()
    return sorted(E)

def f1_aretes(pred, vrai):
    P, V = set(map(tuple, pred)), set(map(tuple, vrai))
    tp = len(P & V)
    p = tp / len(P) if P else 0.0
    r = tp / len(V) if V else 0.0
    return round(2 * p * r / (p + r), 3) if p + r else 0.0

print(f'{"graphe":12s} {"flèches":>8s} {"acyclique":>10s} {"F1 vs expert":>13s}')
for t in ['expert', 'E1', 'E2', 'E3', 'E4', 'E5', 'aleatoire', 'aucun']:
    E = aretes_expert if t == 'expert' else corrompre(aretes_expert, t, 0.3)
    print(f'{t:12s} {len(E):8d} {str(nx.is_directed_acyclic_graph(nx.DiGraph(E))) if E else "-":>10s} {f1_aretes(E, aretes_expert):13.3f}')

In [ ]:
# Cellule 6 — Entraînement avec arrêt anticipé sur la validation
def fixer_graine(g):
    random.seed(g); np.random.seed(g); torch.manual_seed(g); torch.cuda.manual_seed_all(g)

@torch.no_grad()
def evaluer(modele, T, batch=256):
    modele.eval()
    Q, R, M = T
    ys, ps = [], []
    for i in range(0, len(Q), batch):
        q, r, m = Q[i:i+batch].to(DEVICE), R[i:i+batch].to(DEVICE), M[i:i+batch].to(DEVICE)
        p = torch.sigmoid(modele(q, r))
        cible = m[:, 1:]
        ys.append(r[:, 1:][cible].cpu().numpy()); ps.append(p[cible].float().cpu().numpy())
    y, p = np.concatenate(ys), np.concatenate(ps)
    return {'AUC': roc_auc_score(y, p), 'ACC': accuracy_score(y, p >= 0.5)}

def entrainer(nom_modele, A=None, fraction=1.0, graine=GRAINE, hp=HP, verbeux=True):
    fixer_graine(graine)
    T_train = tenseurs(*sous_ensemble_train(fraction, graine))
    modele = (DKT(N, hp) if nom_modele == 'DKT' else GraphKT(N, hp, A)).to(DEVICE)
    opt = torch.optim.Adam(modele.parameters(), lr=hp['lr'])
    perte = nn.BCEWithLogitsLoss()
    Q, R, M = T_train
    meilleur, etat, attente, t0 = -1, None, 0, time.time()
    for ep in range(hp['epoques_max']):
        modele.train()
        ordre = torch.randperm(len(Q))
        for i in range(0, len(Q), hp['batch']):
            idx = ordre[i:i + hp['batch']]
            q, r, m = Q[idx].to(DEVICE), R[idx].to(DEVICE), M[idx].to(DEVICE)
            cible = m[:, 1:]
            l = perte(modele(q, r)[cible], r[:, 1:][cible].float())
            opt.zero_grad(); l.backward()
            nn.utils.clip_grad_norm_(modele.parameters(), 5.0)
            opt.step()
        v = evaluer(modele, T_valid)['AUC']
        if verbeux:
            print(f'   époque {ep+1:2d} | AUC valid {v:.4f} | {(time.time()-t0)/60:.1f} min')
        if v > meilleur + 1e-4:
            meilleur, etat, attente = v, copy.deepcopy(modele.state_dict()), 0
        else:
            attente += 1
            if attente >= hp['patience']:
                break
    modele.load_state_dict(etat)
    test = evaluer(modele, T_test)
    return {'AUC_valid': round(meilleur, 4), 'AUC_test': round(test['AUC'], 4),
            'ACC_test': round(test['ACC'], 4), 'epoques': ep + 1,
            'minutes': round((time.time() - t0) / 60, 1)}, modele

print('OK')

In [ ]:
# Cellule 7 — Test de contrôle : 2 régimes × (DKT + GraphKT avec 3 graphes), 1 graine
CONDITIONS = [('DKT', '-'),
              ('GraphKT', 'expert'),
              ('GraphKT', 'aucun'),
              ('GraphKT', 'aleatoire')]
FRACTIONS = [1.0, 0.1]

chemin_res = os.path.join(RESULTATS, 'phase2_controle.csv')
deja = pd.read_csv(chemin_res) if os.path.exists(chemin_res) else pd.DataFrame()
resultats = deja.to_dict('records')
fait = {(x['modele'], str(x['graphe']), float(x['fraction'])) for x in resultats}

for frac in FRACTIONS:
    for nom, g in CONDITIONS:
        if (nom, str(g), float(frac)) in fait:
            print(f'déjà fait : {nom} / {g} / {frac}'); continue
        print(f'\n>>> {nom} | graphe = {g} | {int(frac*100)} % des élèves')
        A = None
        if nom == 'GraphKT':
            A = A_expert if g == 'expert' else vers_matrice(corrompre(aretes_expert, g, 0))
        res, _ = entrainer(nom, A, fraction=frac)
        resultats.append({'modele': nom, 'graphe': g, 'fraction': frac, 'graine': GRAINE, **res})
        pd.DataFrame(resultats).to_csv(chemin_res, index=False)   # sauvegarde après chaque entraînement
        print('   ->', res)

In [ ]:
# Cellule 8 — Résumé à copier dans la conversation
tab = pd.DataFrame(resultats)
print(tab[['fraction', 'modele', 'graphe', 'AUC_test', 'ACC_test', 'AUC_valid', 'epoques', 'minutes']]
      .sort_values(['fraction', 'modele', 'graphe'], ascending=[False, True, True]).to_string(index=False))

for frac in sorted(tab.fraction.unique(), reverse=True):
    g = tab[(tab.fraction == frac) & (tab.modele == 'GraphKT')].set_index('graphe')['AUC_test']
    if {'expert', 'aucun', 'aleatoire'} <= set(g.index):
        print(f'\n{int(frac*100)} % des élèves : expert − aucun = {g["expert"] - g["aucun"]:+.4f} | '
              f'expert − aléatoire = {g["expert"] - g["aleatoire"]:+.4f}')